In [11]:
taxa_livre_risco_anual = 0.05

sharpe_ratios = {}
sharpe_ratios_diversificados = {}

def calculate_sharpe_ratio(portfolio_data, portfolio_returns_dict, risk_free_rate):
    sharpe_results = {}
    for nome, portfolio in portfolio_data.items():
        portfolio_returns_list = []
        for acao in portfolio:
            if acao in retornos and not retornos[acao].empty:
                # retornos[acao] já é uma série pandas com 'Date' como índice e o ticker como nome.
                # Não é necessário resetar o índice ou renomear, está pronto para ser concatenado.
                portfolio_returns_list.append(retornos[acao])

        if portfolio_returns_list:
            # Concatena os retornos, lidando com possíveis incompatibilidades de índice com outer join
            portfolio_df = pd.concat(portfolio_returns_list, axis=1, join='outer')

            # Calcula os retornos mensais do portfólio (pesos iguais)
            # Remove linhas onde todos os valores são NaN para evitar problemas com o cálculo da média
            portfolio_monthly_returns = portfolio_df.mean(axis=1).dropna()

            if not portfolio_monthly_returns.empty:
                # Calcula o desvio padrão mensal
                std_mensal_portfolio = portfolio_monthly_returns.std()

                # Anualiza o desvio padrão
                volatilidade_anualizada_portfolio = std_mensal_portfolio * np.sqrt(12)

                # Obtém o retorno anualizado do portfólio
                retorno_mensal_medio = portfolio_returns_dict[nome]
                retorno_anualizado_portfolio = retorno_mensal_medio * 12

                # Calcula o Sharpe Ratio
                if volatilidade_anualizada_portfolio > 0:
                    sharpe_ratio = (retorno_anualizado_portfolio - risk_free_rate) / volatilidade_anualizada_portfolio
                    sharpe_results[nome] = sharpe_ratio
                else:
                    sharpe_results[nome] = np.nan # Volatilidade é zero ou negativa, Sharpe é indefinido
            else:
                sharpe_results[nome] = np.nan # Não há retornos mensais suficientes para calcular
        else:
            sharpe_results[nome] = np.nan # Não há retornos de ações válidos para este portfólio
    return sharpe_results

# Calcula o Sharpe Ratio para os portfólios originais
sharpe_ratios = calculate_sharpe_ratio(portfolios, retornos_portfolios, taxa_livre_risco_anual)

# Calcula o Sharpe Ratio para os portfólios diversificados
sharpe_ratios_diversificados = calculate_sharpe_ratio(portfolios_diversificados, retornos_portfolios_diversificados, taxa_livre_risco_anual)

print("Portfólios:")
for nome, portfolio in portfolios.items():
    print(f"{nome}: {portfolio}")

print("\nPortfólios Diversificados:")
for nome, portfolio in portfolios_diversificados.items():
    print(f"{nome}: {portfolio}")

print("\nRetornos Mensais dos Portfólios:")
for nome, retorno in retornos_portfolios.items():
    if not np.isnan(retorno):
        print(f"{nome}: {retorno:.2%}")
    else:
        print(f"{nome}: N/A (dados insuficientes)")

print("\nRetornos Mensais dos Portfólios Diversificados:")
for nome, retorno in retornos_portfolios_diversificados.items():
    if not np.isnan(retorno):
        print(f"{nome}: {retorno:.2%}")
    else:
        print(f"{nome}: N/A (dados insuficientes)")

print("\nSharpe Ratios (Portfólios Originais):")
for nome, ratio in sharpe_ratios.items():
    if not np.isnan(ratio):
        print(f"{nome}: {ratio:.2f}")
    else:
        print(f"{nome}: N/A (dados insuficientes ou volatilidade zero)")

print("\nSharpe Ratios (Portfólios Diversificados):")
for nome, ratio in sharpe_ratios_diversificados.items():
    if not np.isnan(ratio):
        print(f"{nome}: {ratio:.2f}")
    else:
        print(f"{nome}: N/A (dados insuficientes ou volatilidade zero)")

Portfólios:
Risco Baixo: ['ITUB4.SA', 'ABEV3.SA', 'BBDC4.SA', 'ITSA4.SA', 'KLBN11.SA', 'RADL3.SA', 'SUZB3.SA']
Risco Médio: ['VALE3.SA', 'BBAS3.SA', 'B3SA3.SA', 'LREN3.SA', 'MRVE3.SA', 'RENT3.SA']
Risco Alto: ['PETR4.SA', 'ENEV3.SA', 'BPAC11.SA', 'CSNA3.SA', 'GGBR4.SA', 'MGLU3.SA', 'PCAR3.SA']

Portfólios Diversificados:
Risco Baixo: ['ITUB4.SA', 'ABEV3.SA', 'KLBN11.SA', 'RADL3.SA']
Risco Médio: ['VALE3.SA', 'BBAS3.SA', 'B3SA3.SA', 'LREN3.SA', 'MRVE3.SA']
Risco Alto: ['PETR4.SA', 'BPAC11.SA', 'ENEV3.SA', 'CSNA3.SA', 'MGLU3.SA']

Retornos Mensais dos Portfólios:
Risco Baixo: 1.18%
Risco Médio: 1.36%
Risco Alto: 1.91%

Retornos Mensais dos Portfólios Diversificados:
Risco Baixo: 1.31%
Risco Médio: 1.21%
Risco Alto: 1.87%

Sharpe Ratios (Portfólios Originais):
Risco Baixo: 0.52
Risco Médio: 0.42
Risco Alto: 0.52

Sharpe Ratios (Portfólios Diversificados):
Risco Baixo: 0.61
Risco Médio: 0.35
Risco Alto: 0.47
